
# 05 v2. Expert system + explanation / conflict layer

Эта версия **не переобучает ни одну модель и не меняет результаты `04`**.

Исправляем только:

1. `confidence`;
2. B12 rule;
3. inflammation rule;
4. conflict logic.

Главный принцип:

```text
unknown != negative
```



## 1. Архитектура этого этапа

```text
calibrated L1 ─┐
               ├─> weighted ensemble ─> prediction/probability
calibrated CB ─┘                         │
                                        ├─> clinical evidence
clinical rules ──────────────────────────┤
                                        ├─> conflicts
                                        ├─> confidence
                                        └─> next-test suggestions
```

### Важное ограничение

Clinical rules здесь **не меняют уже полученный диагноз**.

Они работают как:

- evidence layer;
- conflict/safety layer;
- explanation layer.

Если позже внешний validation покажет, что rules помогают устойчивости, архитектуру можно пересмотреть.


In [1]:

from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)


## 2. Пути проекта и загрузка зафиксированных OOF predictions

In [2]:

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p.resolve()
    return None

DATA_PATH = first_existing([
    "../data/raw/deficiency_anemia.csv",
    "data/raw/deficiency_anemia.csv",
    "../../data/raw/deficiency_anemia.csv",
])

ENSEMBLE_OOF_PATH = first_existing([
    "../artifacts/ensemble/ensemble_oof_predictions.csv",
    "artifacts/ensemble/ensemble_oof_predictions.csv",
    "../../artifacts/ensemble/ensemble_oof_predictions.csv",
])

if DATA_PATH is None:
    raise FileNotFoundError("Не найден data/raw/deficiency_anemia.csv")

if ENSEMBLE_OOF_PATH is None:
    raise FileNotFoundError(
        "Не найден ensemble_oof_predictions.csv. "
        "Сначала запусти 04_rules_calibration_ensemble_frozen.ipynb полностью."
    )

df = pd.read_csv(DATA_PATH)
oof = pd.read_csv(ENSEMBLE_OOF_PATH)

print("Dataset:", DATA_PATH, df.shape)
print("OOF:", ENSEMBLE_OOF_PATH, oof.shape)


Dataset: /home/sinopah/meditron/data/raw/deficiency_anemia.csv (840, 48)
OOF: /home/sinopah/meditron/artifacts/ensemble/ensemble_oof_predictions.csv (840, 30)



## 3. Подключаем `clinical_rules.py`


```text
meditron/
└── ml/
    ├── clinical_rules.py
```

`04` не изменяется.


In [3]:
from pathlib import Path

path = Path("/home/sinopah/meditron/ml/clinical_rules.py")

print("Файл существует:", path.exists())
print("Размер:", path.stat().st_size)

text = path.read_text(encoding="utf-8")

print(
    "Есть predicted_state_confidence:",
    "def predicted_state_confidence" in text
)

print(
    "Есть confidence_label:",
    "def confidence_label" in text
)

print(
    "Есть B12 v2:",
    "B12 v2" in text
)

print(
    "Есть Inflammation v2:",
    "Inflammation v2" in text
)

Файл существует: True
Размер: 16879
Есть predicted_state_confidence: True
Есть confidence_label: True
Есть B12 v2: True
Есть Inflammation v2: True


In [4]:

module_dir = first_existing([
    "../ml",
    "ml",
    "../../ml",
])

if module_dir is None:
    raise FileNotFoundError(
        "Не найдена папка ml. "
        "Положи clinical_rules.py в meditron/ml/."
    )

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from clinical_rules import (
    CLINICAL,
    RULE_TARGETS,
    anemia_rule,
    compute_rule_scores,
    build_evidence,
    build_conflicts,
    recommended_next_tests,
    predicted_state_confidence,
    confidence_label,
)

print("Clinical rules imported successfully.")


Clinical rules imported successfully.


## 4. Проверяем согласованность пациентов

In [5]:

assert df["patient_id"].is_unique
assert oof["patient_id"].is_unique

work = df.merge(
    oof,
    on="patient_id",
    how="inner",
    validate="one_to_one",
)

assert len(work) == len(df) == len(oof), (
    len(df),
    len(oof),
    len(work),
)

print("Patients matched:", len(work))


Patients matched: 840



## 5. Формируем explanation payload для одного пациента

Payload состоит из:

```text
patient_id
prediction
confidence
deficiencies
evidence
conflicts
recommended_next_tests
```

`confidence` — это не вероятность «истинности диагноза».
Это удобная продуктовая категория на основе максимальной ensemble probability.


In [6]:

def patient_probabilities(row):
    return {
        target: float(row[f"{target}__probability"])
        for target in RULE_TARGETS
    }


def patient_predictions(row):
    return {
        target: int(row[f"{target}__prediction"])
        for target in RULE_TARGETS
    }


def patient_rule_scores(row):
    return compute_rule_scores(row)


def branch_confidence(probability, prediction):
    certainty = predicted_state_confidence(
        probability=probability,
        prediction=prediction,
    )

    return {
        "selected_state": "positive" if int(prediction) == 1 else "negative",
        "certainty": round(certainty, 4),
        "level": confidence_label(certainty),
    }


def overall_screening_confidence(probabilities, predictions):
    positive_targets = [
        target
        for target in RULE_TARGETS
        if predictions[target] == 1
    ]

    if positive_targets:
        certainties = {
            target: predicted_state_confidence(
                probabilities[target],
                predictions[target],
            )
            for target in positive_targets
        }

        limiting_target = min(
            certainties,
            key=certainties.get,
        )
        certainty = certainties[limiting_target]

        return {
            "basis": "weakest_positive_branch",
            "limiting_target": limiting_target,
            "certainty": round(certainty, 4),
            "level": confidence_label(certainty),
        }

    strongest_possible_deficiency = max(
        probabilities,
        key=probabilities.get,
    )
    certainty = 1.0 - probabilities[strongest_possible_deficiency]

    return {
        "basis": "all_binary_branches_negative",
        "limiting_target": strongest_possible_deficiency,
        "certainty": round(certainty, 4),
        "level": confidence_label(certainty),
    }


def build_explanation_payload(row):
    probabilities = patient_probabilities(row)
    predictions = patient_predictions(row)
    rule_scores = patient_rule_scores(row)

    deficiencies = {}

    for target in RULE_TARGETS:
        deficiencies[target] = {
            "probability": round(probabilities[target], 4),
            "prediction": bool(predictions[target]),
            "rule_score": round(rule_scores[target], 4),
            "confidence": branch_confidence(
                probabilities[target],
                predictions[target],
            ),
        }

    payload = {
        "patient_id": row["patient_id"],
        "prediction": {
            "anemia_class": row["ensemble_class_pred"],
            "deficiency_cause": row["ensemble_cause_pred"],
            "anemia": bool(row["anemia_rule"]),
        },
        "confidence": overall_screening_confidence(
            probabilities=probabilities,
            predictions=predictions,
        ),
        "deficiencies": deficiencies,
        "evidence": build_evidence(row),
        "conflicts": build_conflicts(
            probabilities=probabilities,
            predictions=predictions,
            rule_scores=rule_scores,
        ),
        "recommended_next_tests": recommended_next_tests(
            row,
            probabilities,
        ),
        "disclaimer": (
            "Decision-support output. "
            "Не является самостоятельным клиническим диагнозом "
            "и требует интерпретации врачом."
        ),
    }

    return payload


## 6. Пример payload

In [7]:

example_payload = build_explanation_payload(work.iloc[0])

print(json.dumps(
    example_payload,
    ensure_ascii=False,
    indent=2,
))


{
  "patient_id": "P000001",
  "prediction": {
    "anemia_class": "B12_deficiency_no_anemia",
    "deficiency_cause": "B12_deficiency",
    "anemia": false
  },
  "confidence": {
    "basis": "weakest_positive_branch",
    "limiting_target": "B12_deficiency",
    "certainty": 0.8282,
    "level": "moderate"
  },
  "deficiencies": {
    "iron_deficiency": {
      "probability": 0.0009,
      "prediction": false,
      "rule_score": 0.05,
      "confidence": {
        "selected_state": "negative",
        "certainty": 0.9991,
        "level": "high"
      }
    },
    "B12_deficiency": {
      "probability": 0.8282,
      "prediction": true,
      "rule_score": 0.5,
      "confidence": {
        "selected_state": "positive",
        "certainty": 0.8282,
        "level": "moderate"
      }
    },
    "folate_deficiency": {
      "probability": 0.1302,
      "prediction": false,
      "rule_score": 0.95,
      "confidence": {
        "selected_state": "negative",
        "certainty": 0.86


## 7. Строим payload для всех пациентов

Это нужно не для обучения, а для проверки:

- нет ли слишком большого количества conflicts;
- где чаще возникают конфликты;
- какие анализы чаще всего предлагаются как недостающие;
- не выдаёт ли explanation layer нелогичные сообщения.


In [8]:

payloads = [
    build_explanation_payload(row)
    for _, row in work.iterrows()
]

print("Payloads:", len(payloads))


Payloads: 840


## 8. Анализ conflicts

In [9]:

conflict_rows = []

for payload in payloads:
    for conflict in payload["conflicts"]:
        conflict_rows.append({
            "patient_id": payload["patient_id"],
            **conflict,
        })

conflict_df = pd.DataFrame(conflict_rows)

if len(conflict_df) == 0:
    print("Сильных конфликтов по текущим product-thresholds нет.")
else:
    display(
        conflict_df
        .groupby(["target", "type"])
        .size()
        .rename("n")
        .reset_index()
        .sort_values("n", ascending=False)
    )

    display(conflict_df.head(20))


,target,type,n
1,inflammation_anemia,ml_negative_rule_positive,40
3,iron_deficiency,ml_positive_rule_negative,22
2,iron_deficiency,ml_negative_rule_positive,12
0,folate_deficiency,ml_negative_rule_positive,7


,patient_id,target,type,severity,ensemble_probability,rule_score,rule_state,message
0,P000001,folate_deficiency,ml_negative_rule_positive,review,0.130243,0.95,supports,"Clinical rule layer сильно поддерживает состояние, а ML-вероятность низкая."
1,P000013,iron_deficiency,ml_positive_rule_negative,review,0.998939,0.05,against,Высокая ML-вероятность при явном clinical evidence против соответствующего состояния.
2,P000016,iron_deficiency,ml_positive_rule_negative,review,0.752516,0.10,against,Высокая ML-вероятность при явном clinical evidence против соответствующего состояния.
3,P000018,inflammation_anemia,ml_negative_rule_positive,review,0.002699,0.95,supports,"Clinical rule layer сильно поддерживает состояние, а ML-вероятность низкая."
4,P000020,inflammation_anemia,ml_negative_rule_positive,review,0.000042,0.95,supports,"Clinical rule layer сильно поддерживает состояние, а ML-вероятность низкая."
5,P000026,iron_deficiency,ml_positive_rule_negative,review,0.999136,0.10,against,Высокая ML-вероятность при явном clinical evidence против соответствующего состояния.
6,P000044,inflammation_anemia,ml_negative_rule_positive,review,0.000054,0.95,supports,"Clinical rule layer сильно поддерживает состояние, а ML-вероятность низкая."
7,P000046,inflammation_anemia,ml_negative_rule_positive,review,0.000172,0.95,supports,"Clinical rule layer сильно поддерживает состояние, а ML-вероятность низкая."
8,P000048,iron_deficiency,ml_negative_rule_positive,review,0.011426,0.75,supports,"Clinical rule layer сильно поддерживает состояние, а ML-вероятность низкая."
9,P000079,folate_deficiency,ml_negative_rule_positive,review,0.152439,0.95,supports,"Clinical rule layer сильно поддерживает состояние, а ML-вероятность низкая."



### Как интерпретировать conflict v2

Clinical rule теперь имеет три состояния:

```text
rule_score >= 0.75  -> supports
rule_score <= 0.25  -> against
иначе               -> unknown
```

Conflict создаётся только при сильном противоречии:

```text
ML strongly positive + rule explicitly against
```

или:

```text
ML strongly negative + rule strongly supports
```

`unknown` не создаёт конфликт.


## 9. Частота рекомендуемых дополнительных анализов

In [10]:

test_rows = []

for payload in payloads:
    for item in payload["recommended_next_tests"]:
        test_rows.append({
            "patient_id": payload["patient_id"],
            **item,
        })

recommended_tests_df = pd.DataFrame(test_rows)

if len(recommended_tests_df) == 0:
    print("Нет рекомендаций по дополнительным анализам.")
else:
    display(
        recommended_tests_df["test"]
        .value_counts()
        .rename_axis("test")
        .reset_index(name="n")
    )


,test,n
0,Ret_He,210
1,homocysteine,159
2,active_B12,101
3,MMA,90
4,TSAT,45
5,ferritin,25
6,vitamin_B12,22
7,ceruloplasmin,7
8,folate,5
9,copper,1



## 10. Проверяем explanation layer на нескольких типичных случаях

Выбираем:

- самый уверенный прогноз;
- самый неуверенный прогноз;
- пациента с конфликтом, если такой есть.

Это ручная sanity-check проверка перед API.


In [11]:

confidence_scores = work.apply(
    lambda row: overall_screening_confidence(
        patient_probabilities(row),
        patient_predictions(row),
    )["certainty"],
    axis=1,
)

most_confident_idx = confidence_scores.idxmax()
least_confident_idx = confidence_scores.idxmin()

examples = {
    "most_confident": build_explanation_payload(
        work.loc[most_confident_idx]
    ),
    "least_confident": build_explanation_payload(
        work.loc[least_confident_idx]
    ),
}

if len(conflict_df) > 0:
    conflict_patient = conflict_df.iloc[0]["patient_id"]
    conflict_row = work.loc[
        work["patient_id"] == conflict_patient
    ].iloc[0]

    examples["with_conflict"] = build_explanation_payload(
        conflict_row
    )

for name, payload in examples.items():
    print("\n" + "=" * 80)
    print(name)
    print(json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
    ))



most_confident
{
  "patient_id": "P000003",
  "prediction": {
    "anemia_class": "latent_deficiency",
    "deficiency_cause": "iron_deficiency",
    "anemia": false
  },
  "confidence": {
    "basis": "weakest_positive_branch",
    "limiting_target": "iron_deficiency",
    "certainty": 1.0,
    "level": "high"
  },
  "deficiencies": {
    "iron_deficiency": {
      "probability": 1.0,
      "prediction": true,
      "rule_score": 0.725,
      "confidence": {
        "selected_state": "positive",
        "certainty": 1.0,
        "level": "high"
      }
    },
    "B12_deficiency": {
      "probability": 0.0,
      "prediction": false,
      "rule_score": 0.5,
      "confidence": {
        "selected_state": "negative",
        "certainty": 1.0,
        "level": "high"
      }
    },
    "folate_deficiency": {
      "probability": 0.022,
      "prediction": false,
      "rule_score": 0.5,
      "confidence": {
        "selected_state": "negative",
        "certainty": 0.978,
        "l


## 10.1. Sanity-check v2

Проверяем:

- сколько осталось B12 conflicts;
- сколько осталось inflammation conflicts;
- сколько B12 cases стали neutral;
- сколько inflammation cases стали neutral;
- исправился ли confidence для уверенных отрицательных прогнозов.


In [12]:

if len(conflict_df) > 0:
    v2_conflict_counts = (
        conflict_df
        .groupby(["target", "type"])
        .size()
        .rename("n")
        .reset_index()
        .sort_values("n", ascending=False)
    )
    display(v2_conflict_counts)
else:
    v2_conflict_counts = pd.DataFrame(
        columns=["target", "type", "n"]
    )
    print("После v2 сильных conflicts не осталось.")

rule_diagnostics = []

for _, row in work.iterrows():
    scores = patient_rule_scores(row)

    rule_diagnostics.append({
        "patient_id": row["patient_id"],
        "B12_rule_score": scores["B12_deficiency"],
        "inflammation_rule_score": scores["inflammation_anemia"],
    })

rule_diagnostics = pd.DataFrame(rule_diagnostics)

print(
    "B12 neutral rule_score == 0.5:",
    int((rule_diagnostics["B12_rule_score"] == 0.5).sum()),
)

print(
    "Inflammation neutral rule_score == 0.5:",
    int((rule_diagnostics["inflammation_rule_score"] == 0.5).sum()),
)

confidence_table = pd.DataFrame([
    {
        "patient_id": payload["patient_id"],
        "confidence": payload["confidence"]["certainty"],
        "level": payload["confidence"]["level"],
        "basis": payload["confidence"]["basis"],
    }
    for payload in payloads
])

display(
    confidence_table["level"]
    .value_counts()
    .rename_axis("level")
    .reset_index(name="n")
)

display(
    confidence_table
    .sort_values("confidence")
    .head(10)
)


,target,type,n
1,inflammation_anemia,ml_negative_rule_positive,40
3,iron_deficiency,ml_positive_rule_negative,22
2,iron_deficiency,ml_negative_rule_positive,12
0,folate_deficiency,ml_negative_rule_positive,7


B12 neutral rule_score == 0.5: 778
Inflammation neutral rule_score == 0.5: 147


,level,n
0,high,695
1,moderate,92
2,low,53


,patient_id,confidence,level,basis
404,P000405,0.1726,low,weakest_positive_branch
363,P000364,0.1773,low,weakest_positive_branch
106,P000107,0.1985,low,weakest_positive_branch
308,P000309,0.2313,low,weakest_positive_branch
379,P000380,0.2400,low,weakest_positive_branch
619,P000620,0.2412,low,weakest_positive_branch
193,P000194,0.2729,low,weakest_positive_branch
131,P000132,0.3158,low,weakest_positive_branch
575,P000576,0.3462,low,weakest_positive_branch
298,P000299,0.3528,low,all_binary_branches_negative


## 11. Сохраняем explanation artifacts

In [13]:

def resolve_explanation_artifact_root():
    candidates = [
        Path("../artifacts/explanations"),
        Path("artifacts/explanations"),
        Path("../../artifacts/explanations"),
    ]
    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate.resolve()
        except Exception:
            pass
    raise RuntimeError("Не удалось создать artifacts/explanations")

EXPLANATION_DIR = resolve_explanation_artifact_root()

# JSONL удобно использовать в API/ETL.
jsonl_path = EXPLANATION_DIR / "explanation_payloads.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for payload in payloads:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

if len(conflict_df) > 0:
    conflict_df.to_csv(
        EXPLANATION_DIR / "conflicts.csv",
        index=False,
    )
else:
    pd.DataFrame(
        columns=[
            "patient_id",
            "target",
            "type",
            "severity",
            "ensemble_probability",
            "rule_score",
            "message",
        ]
    ).to_csv(
        EXPLANATION_DIR / "conflicts.csv",
        index=False,
    )

if len(recommended_tests_df) > 0:
    recommended_tests_df.to_csv(
        EXPLANATION_DIR / "recommended_next_tests.csv",
        index=False,
    )

print("Saved to:", EXPLANATION_DIR)
for p in sorted(EXPLANATION_DIR.iterdir()):
    print(" -", p.name)


Saved to: /home/sinopah/meditron/artifacts/explanations
 - conflicts.csv
 - explanation_payloads.jsonl
 - recommended_next_tests.csv



# 12. Что прислать после запуска v2

Пришли:

1. `v2_conflict_counts`;
2. строки:
   - `B12 neutral rule_score == 0.5`
   - `Inflammation neutral rule_score == 0.5`
3. распределение `confidence` по `high / moderate / low`;
4. нижние 10 строк `confidence_table`.

Ожидаем:

- резкое снижение искусственных B12 conflicts;
- резкое снижение искусственных inflammation conflicts;
- уверенные отрицательные прогнозы больше не должны автоматически иметь `low confidence`.

Если это подтверждается, переходим к `06_external_validation.ipynb`.
